# Mark a protein as prepared

Stamp a local PDB or mmCIF so downstream tools treat it as already prepared.
`mark_as_prepared()` mutates the on-disk file in its native format (no CIF→PDB conversion).
Call `sync()` separately to upload the stamped bytes to UFA.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import tempfile

from deeporigin.drug_discovery import Protein
from deeporigin.drug_discovery.structures.prepared_protein_stamp import (
    has_prepared_protein_stamp,
)
from deeporigin.utils.constants import PREPARED_PROTEIN_CIF_STAMP_LINE

workdir = Path(tempfile.mkdtemp(prefix="mark-prepared-"))
pdb_path = workdir / "ala.pdb"
pdb_path.write_text(
    "ATOM      1  N   ALA A   1      11.104  13.207   9.068  1.00  0.00           N\n"
    "END\n"
)

protein = Protein.from_file(pdb_path)
protein.mark_as_prepared()
assert has_prepared_protein_stamp(pdb_path)
print("Stamped PDB:", pdb_path.read_text().splitlines()[0])

In [ ]:
cif_src = Path("tests/fixtures/1EBY.cif")
cif_path = workdir / "1eby.cif"
cif_path.write_text(cif_src.read_text())

protein_cif = Protein.from_file(cif_path)
protein_cif.mark_as_prepared()
assert has_prepared_protein_stamp(cif_path)
assert PREPARED_PROTEIN_CIF_STAMP_LINE in cif_path.read_text()
print("CIF remains .cif:", cif_path.suffix)
print("Stamp line present:", PREPARED_PROTEIN_CIF_STAMP_LINE)

To push stamped bytes to the platform (overwriting an existing UFA object when
`remote_path` is set), call `protein.sync()` after marking. That upload uses the
local file bytes and does not convert CIF to PDB.